# 01 · Data Loading & Exploratory Analysis
Fetches D2 and 5-HT2A bioactivity data from ChEMBL, curates structures,
computes Morgan fingerprints and Murcko scaffolds, and saves `data.pkl`
for downstream notebooks.

**Output:** `data.pkl`

In [ ]:
import sys, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '.')
from utils import (fetch_curate, get_fp, get_scaffold, SEED, N_FP)
from tqdm import tqdm
tqdm.pandas()

np.random.seed(SEED)
print("Imports OK.")

## 1. Fetch & curate from ChEMBL
Targets: D2 (`CHEMBL217`) and 5-HT2A (`CHEMBL224`).

In [ ]:
# ── Extension hook: add receptor pairs here ───────────────────────────────────
TARGETS = {
    'D2':    'CHEMBL217',
    '5HT2A': 'CHEMBL224',
    # 'H1':  'CHEMBL231',   # uncomment to extend
    # 'D2':  'CHEMBL217',   # add more pairs as needed
}

print("Fetching activity data from ChEMBL...")
datasets = {}
for name, cid in TARGETS.items():
    datasets[name] = fetch_curate(name, cid)

d2  = datasets['D2']
sht = datasets['5HT2A']

## 2. Fingerprints & scaffolds

In [ ]:
print("Computing fingerprints and scaffolds...")
for df in [d2, sht]:
    df['fp']       = df['curated_smiles'].progress_apply(get_fp)
    df['scaffold'] = df['curated_smiles'].apply(get_scaffold)
    df.dropna(subset=['fp'], inplace=True)

# Overlap dataset — compounds tested at BOTH receptors
merged = d2.merge(sht, on='curated_smiles', suffixes=('_D2','_5HT2A'))
merged['delta']    = merged['pChEMBL_5HT2A'] - merged['pChEMBL_D2']
merged['fp']       = merged['curated_smiles'].progress_apply(get_fp)
merged['scaffold'] = merged['curated_smiles'].apply(get_scaffold)
merged.dropna(subset=['fp'], inplace=True)

# Raw fingerprint arrays
X_d2  = np.stack(d2['fp'].values)
X_sht = np.stack(sht['fp'].values)
X_ov  = np.stack(merged['fp'].values)

print(f"D2:      {len(d2):,} unique structures")
print(f"5HT2A:   {len(sht):,} unique structures")
print(f"Overlap: {len(merged):,} compounds tested at both receptors")
print(f"Atypical (5HT2A > D2): {(merged['delta'] > 0).mean()*100:.1f}% of overlap")

## 3. Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# 3A. Activity distributions
axes[0,0].hist(merged['pChEMBL_D2'],    bins=40, alpha=0.7, color='#7F77DD', label='D2')
axes[0,0].hist(merged['pChEMBL_5HT2A'], bins=40, alpha=0.7, color='#D85A30', label='5HT2A')
axes[0,0].set(xlabel='pChEMBL', title='Activity distributions (overlap set)')
axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

# 3B. Atypicality ratio
axes[0,1].hist(merged['delta'], bins=40, color='#1D9E75', edgecolor='black', alpha=0.8)
axes[0,1].axvline(0, color='red', lw=2, linestyle='--', label='Δ = 0')
axes[0,1].set(xlabel='ΔpChEMBL (5HT2A − D2)',
              title='Atypicality ratio
(+ve = atypical profile)')
axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

# 3C. D2 vs 5HT2A scatter
sc = axes[0,2].scatter(merged['pChEMBL_D2'], merged['pChEMBL_5HT2A'],
                        c=merged['delta'], cmap='RdBu_r', s=12, alpha=0.5)
lo = min(merged['pChEMBL_D2'].min(), merged['pChEMBL_5HT2A'].min()) - 0.3
hi = max(merged['pChEMBL_D2'].max(), merged['pChEMBL_5HT2A'].max()) + 0.3
axes[0,2].plot([lo,hi],[lo,hi],'k--',lw=1,alpha=0.5)
axes[0,2].set(xlabel='pChEMBL D2', ylabel='pChEMBL 5HT2A',
              title='Per-compound activities')
plt.colorbar(sc, ax=axes[0,2], label='ΔpChEMBL')

# 3D. pChEMBL by assay type
for name, df_r, ax in [(('D2', d2, axes[1,0])), ('5HT2A', sht, axes[1,1])]:
    pass  # placeholder — add assay breakdown if standard_type retained

# 3D. Dataset sizes
sizes = {'D2
(all)': len(d2), '5HT2A
(all)': len(sht),
         'Overlap': len(merged)}
axes[1,0].bar(sizes.keys(), sizes.values(),
              color=['#7F77DD','#D85A30','#1D9E75'], alpha=0.85, edgecolor='black')
for k, v in zip(sizes.keys(), sizes.values()):
    axes[1,0].text(list(sizes.keys()).index(k), v+30, f'{v:,}', ha='center', fontsize=10)
axes[1,0].set(title='Dataset sizes', ylabel='Compounds')
axes[1,0].grid(alpha=0.3, axis='y')

# 3E. pChEMBL statistics table
stats = merged[['pChEMBL_D2','pChEMBL_5HT2A','delta']].describe().round(2)
axes[1,1].axis('off')
tbl = axes[1,1].table(cellText=stats.values, rowLabels=stats.index,
                       colLabels=['D2','5HT2A','Δ'],
                       loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(9)
axes[1,1].set_title('pChEMBL statistics', fontweight='bold')

# 3F. Scaffold diversity
sc_counts = merged['scaffold'].value_counts()
axes[1,2].hist(sc_counts.values, bins=30, color='#534AB7', edgecolor='black', alpha=0.8)
axes[1,2].set(xlabel='Compounds per scaffold', ylabel='Number of scaffolds',
              title=f'Scaffold diversity\n{len(sc_counts):,} unique scaffolds')
axes[1,2].grid(alpha=0.3)

plt.suptitle('D2 / 5HT2A Dataset Overview', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_01_eda.png', dpi=130, bbox_inches='tight')
plt.show()

## 4. Save data.pkl

In [ ]:
data = {
    'd2':     d2,
    'sht':    sht,
    'merged': merged,
    'X_d2':   X_d2,
    'X_sht':  X_sht,
    'X_ov':   X_ov,
}

with open('data.pkl','wb') as f:
    pickle.dump(data, f)

print("Saved: data.pkl")
print(f"  Keys: {list(data.keys())}")
print(f"  X_d2:  {X_d2.shape}  X_sht: {X_sht.shape}  X_ov: {X_ov.shape}")